# Dataset overview (Hub sanity check)

**This notebook does not build datasets.** It loads published splits from [Pawlo77/mllm-shap](https://huggingface.co/datasets/Pawlo77/mllm-shap) and checks that rows, columns, and audio bytes look correct before heavy experiments.

## Purpose

After uploading parquets from the builder notebooks (`voice_bench.ipynb`, `infinity_instruct.ipynb`, LibriSpeech builder), use this notebook to confirm the pinned Hub revision still matches what downstream code expects: schema, row counts, and playable TTS audio.

Hub configs use `{task}__{source}` names (see `experiments/data_preparation/hf/README.md`):

| Config | Source | Notes (last publish) |
|--------|--------|----------------------|
| `single_sentence__voice_bench` | VoiceBench | 854 rows |
| `multi_sentence__voice_bench` | VoiceBench | 103 rows |
| `single_sentence__librispeech_asr` | LibriSpeech ASR | 609 rows (target 1k) |
| `multi_lingual__infinity_instruct` | Infinity-Instruct | 435 rows (145 × 3 languages) |

This notebook:

1. Optionally lists row counts for every config at `REVISION`.
2. Loads one config's `test` split (random or set `CONFIG_NAME`).
3. Prints the `DatasetDict` and a random row's field keys.
4. Plays the first clip from `audio__male` (or `audio__female` on `multi_lingual__infinity_instruct`).
5. Prints `sentences` (or legacy `prompt`).

Pin `REVISION` after upload: `make -C experiments/data_preparation revision` (prints the latest commit hash).

Use for **quick listen-and-inspect** checks—not to filter, sample, synthesize, or upload.

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
from src.setup import configure_notebook_environment

configure_notebook_environment(set_plot_style=True)

PosixPath('/Users/pawelp/Desktop/education/pw/bachelor/MLLM-Shap')

In [3]:
from pprint import pprint

from src.constants import HUB_REPO_ID
from src.hub_overview import (
    HUB_OVERVIEW_CONFIGS,
    PREFER_FEMALE_AUDIO_CONFIGS,
    inspect_random_test_row,
    load_hub_dataset,
    pick_random_config,
    play_first_audio_clip,
    show_text_content,
    summarize_hub_configs,
)

DATASET_NAME: str = HUB_REPO_ID
REVISION: str = "main"  # update: make revision

# Set to a Hub config name to skip random pick, e.g. SINGLE_SENTENCE__VOICE_BENCH
CONFIG_NAME: str | None = None

# Set True to print row counts for every config (loads each split from the Hub)
SUMMARIZE_ALL_CONFIGS = False

if SUMMARIZE_ALL_CONFIGS:
    summarize_hub_configs(REVISION, dataset_name=DATASET_NAME)

config = CONFIG_NAME or pick_random_config(HUB_OVERVIEW_CONFIGS)
ds = load_hub_dataset(DATASET_NAME, config, REVISION)

pprint(ds)

Using config: multi_sentence__voice_bench


README.md:   0%|          | 0.00/7.70k [00:00<?, ?B/s]

I0522 22:18:52.702527 3998264 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


multi_sentence__voice_bench/test/0000.pa(…):   0%|          | 0.00/35.7M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/103 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['datasets', 'sentences', 'sentences__num', 'token_count', 'audio__original', 'audio__original__duration', 'audio__male', 'audio__male__duration', 'audio__female', 'audio__female__duration'],
        num_rows: 103
    })
})


In [4]:
sample_entry = inspect_random_test_row(ds)

dict_keys(['datasets', 'sentences', 'sentences__num', 'token_count', 'audio__original', 'audio__original__duration', 'audio__male', 'audio__male__duration', 'audio__female', 'audio__female__duration'])


In [5]:
play_first_audio_clip(sample_entry, prefer_female=config in PREFER_FEMALE_AUDIO_CONFIGS)

In [6]:
show_text_content(sample_entry)

['Come up with three names for a second base software company.',
 'Make sure your names are in English and all capital letters.']
